# Project 03 — Four Linear Algebra Methods

## Notebook 03 — Singular Value Decomposition (SVD)

**Blocks:** 11–15  
**Dataset:** 400 MRI images; 128 × 128 grayscale; 16,384 features per image  
**Main project directory (Google Colab):** `MyDrive/MATHEMATICS_PROJECTS/PROJECT_03_LA_ML_FOUR_METHODS/`  
**Processed files:** `data/processed/X_mri_400.npy`, `y_mri_400.npy`, `filenames_mri_400.json`

*Execution note: Run cells in order after mounting Google Drive. Notebook 01 creates the processed dataset used by the following notebooks.*


### Mathematical purpose and interpretation

Singular Value Decomposition (SVD) separates the matrix into orthogonal left and right singular directions with nonnegative singular values. Keeping the first $k$ singular directions yields a rank-$k$ approximation to the matrix and provides a principled compression-reconstruction comparison.

For the matrix used here, the singular-value energy thresholds are reached with **44, 106, and 257** directions for **90%, 95%, and 99% energy**, respectively; corresponding relative reconstruction errors are approximately **0.3150, 0.2232, and 0.0996**. These results concern the **uncentered matrix**, so its energy values should not be conflated with the PCA explained variance of the centered matrix. A nearly exact full-rank reconstruction verifies the decomposition numerically; it does not establish predictive performance.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Block 11 — Mri Matrix Loading And Verification


In [2]:

from pathlib import Path
import numpy as np

# Project directory
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "MATHEMATICS_PROJECTS/"
    "PROJECT_03_LA_ML_FOUR_METHODS"
)

print("BLOCK 11 — MRI MATRIX LOADING AND VERIFICATION")

# Discover previously saved NumPy matrices
npy_files = sorted(PROJECT_ROOT.rglob("*.npy"))
npz_files = sorted(PROJECT_ROOT.rglob("*.npz"))

print("\nPROJECT DIRECTORY")
print("Project root:", PROJECT_ROOT)
print("Directory exists:", PROJECT_ROOT.exists())

print("\nAVAILABLE NUMPY FILES")
for path in npy_files + npz_files:
    print(path.relative_to(PROJECT_ROOT))

# Verify that previous data are available
assert PROJECT_ROOT.exists(), "Project directory not found."
assert len(npy_files) + len(npz_files) > 0, (
    "No saved NumPy files found. "
    "Check the previous PCA notebook."
)

print("\nBLOCK 11 — FILE DISCOVERY PASSED")


BLOCK 11 — MRI MATRIX LOADING AND VERIFICATION

PROJECT DIRECTORY
Project root: /content/drive/MyDrive/MATHEMATICS_PROJECTS/PROJECT_03_LA_ML_FOUR_METHODS
Directory exists: True

AVAILABLE NUMPY FILES
data/processed/X_mri_400.npy
data/processed/y_mri_400.npy

BLOCK 11 — FILE DISCOVERY PASSED


# ***Block 11 — Final verification***

In [3]:

import numpy as np

# Load the same MRI dataset used in previous notebooks
DATA_DIR = PROJECT_ROOT / "data" / "processed"

X = np.load(DATA_DIR / "X_mri_400.npy")
y = np.load(DATA_DIR / "y_mri_400.npy")

# Use float64 for numerical verification
X64 = X.astype(np.float64)

n, p = X64.shape

# Numerical checks
finite_values = np.isfinite(X64).all()
finite_labels = np.isfinite(y).all()
matching_samples = len(y) == n

# Summary statistics
print("BLOCK 11 — MRI MATRIX VERIFICATION")

print("\nMATRIX DIMENSIONS")
print("X shape:", X64.shape)
print("y shape:", y.shape)
print("Number of images:", n)
print("Features per image:", p)

print("\nNUMERICAL PROPERTIES")
print("Data type:", X64.dtype)
print("Minimum value:", X64.min())
print("Maximum value:", X64.max())
print("Mean value:", X64.mean())
print("Standard deviation:", X64.std())

print("\nCLASS DISTRIBUTION")
classes, counts = np.unique(y, return_counts=True)
for cls, count in zip(classes, counts):
    print(f"Class {cls}: {count} images")

print("\nFINAL VERIFICATION")
checks = {
    "Matrix is two-dimensional": X64.ndim == 2,
    "All matrix values are finite": finite_values,
    "All labels are finite": finite_labels,
    "Samples and labels match": matching_samples,
    "Matrix dimensions are positive": n > 0 and p > 0,
}

for name, passed in checks.items():
    print(f"{name}: {passed}")

assert all(checks.values()), "Block 11 verification failed."

print("\nALL BLOCK 11 CHECKS PASSED:", all(checks.values()))


BLOCK 11 — MRI MATRIX VERIFICATION

MATRIX DIMENSIONS
X shape: (400, 16384)
y shape: (400,)
Number of images: 400
Features per image: 16384

NUMERICAL PROPERTIES
Data type: float64
Minimum value: 0.0
Maximum value: 1.0
Mean value: 0.18548444827339708
Standard deviation: 0.1962425453752799

CLASS DISTRIBUTION
Class 0: 100 images
Class 1: 100 images
Class 2: 100 images
Class 3: 100 images

FINAL VERIFICATION
Matrix is two-dimensional: True
All matrix values are finite: True
All labels are finite: True
Samples and labels match: True
Matrix dimensions are positive: True

ALL BLOCK 11 CHECKS PASSED: True



### Block 11 — MRI Matrix Verification

**Objective:** Load and validate the MRI feature matrix before applying Singular Value Decomposition (SVD).

**Dataset:**
- 400 MRI images.
- 16,384 numerical features per image.
- Four balanced classes (100 images each).
- Normalized pixel intensities in [0, 1].

**Matrix representation:**

X ∈ R^(400 × 16384)

**Verification:**
- All matrix values are finite.
- Class labels match the 400 observations.
- Matrix dimensions are valid.
- Numerical precision: float64.

**Conclusion:**

The MRI feature matrix was successfully loaded and verified. It is suitable for SVD analysis and subsequent low-rank approximation experiments.

**Status:** All Block 11 checks passed.


## Block 12 — Singular Value Decomposition


In [4]:

import numpy as np

print("BLOCK 12 — SINGULAR VALUE DECOMPOSITION")

# Matrix from Block 11
X_svd = X64

n, p = X_svd.shape

# Smaller Gram matrix
G = X_svd @ X_svd.T

# Symmetric eigendecomposition
eigenvalues, U = np.linalg.eigh(G)

# Sort eigenvalues in descending order
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
U = U[:, order]

# Singular values
singular_values = np.sqrt(np.maximum(eigenvalues, 0.0))

# Effective numerical rank
tolerance = (
    max(n, p)
    * np.finfo(np.float64).eps
    * singular_values[0]
)

rank = np.sum(singular_values > tolerance)

# Verify Gram eigendecomposition
G_reconstructed = (
    U @ np.diag(eigenvalues) @ U.T
)

gram_error = (
    np.linalg.norm(G - G_reconstructed, ord="fro")
    / np.linalg.norm(G, ord="fro")
)

# Verify orthogonality
orthogonality_error = np.linalg.norm(
    U.T @ U - np.eye(n),
    ord="fro"
)

print("\nMATRIX DIMENSIONS")
print("X shape:", X_svd.shape)
print("Gram matrix shape:", G.shape)
print("U shape:", U.shape)

print("\nSINGULAR VALUES")
print("Number of singular values:", len(singular_values))
print("Largest singular value:", singular_values[0])
print("Smallest singular value:", singular_values[-1])
print("First 10 singular values:", singular_values[:10])

print("\nRANK")
print("Numerical rank:", rank)
print("Maximum possible rank:", min(n, p))

print("\nNUMERICAL VERIFICATION")
print("Relative Gram reconstruction error:", gram_error)
print("U orthogonality error:", orthogonality_error)

checks = {
    "Singular values are nonnegative":
        np.all(singular_values >= 0),
    "Singular values are descending":
        np.all(np.diff(singular_values) <= 0),
    "Gram reconstruction verified":
        gram_error < 1e-10,
    "U orthogonality verified":
        orthogonality_error < 1e-10,
    "Rank within theoretical bounds":
        0 <= rank <= min(n, p),
}

for name, passed in checks.items():
    print(f"{name}: {passed}")

assert all(checks.values()), (
    "Block 12 verification failed."
)

print("\nALL BLOCK 12 CHECKS PASSED:", all(checks.values()))


BLOCK 12 — SINGULAR VALUE DECOMPOSITION

MATRIX DIMENSIONS
X shape: (400, 16384)
Gram matrix shape: (400, 400)
U shape: (400, 400)

SINGULAR VALUES
Number of singular values: 400
Largest singular value: 591.1514360049304
Smallest singular value: 0.15509554988192492
First 10 singular values: [591.151436   145.62399988  74.96955864  70.69436029  53.51830317
  52.54073475  50.5976185   48.04749619  45.92240105  43.54731148]

RANK
Numerical rank: 400
Maximum possible rank: 400

NUMERICAL VERIFICATION
Relative Gram reconstruction error: 1.549555187511749e-15
U orthogonality error: 4.7103612361882906e-14
Singular values are nonnegative: True
Singular values are descending: True
Gram reconstruction verified: True
U orthogonality verified: True
Rank within theoretical bounds: True

ALL BLOCK 12 CHECKS PASSED: True



### Block 12 — Singular Value Decomposition

**Objective:** Compute the singular values and numerical rank of the MRI feature matrix.

**Matrix dimensions:**
- X: 400 × 16,384
- Gram matrix XXᵀ: 400 × 400
- Left singular vector matrix U: 400 × 400

**Results:**
- Largest singular value: 591.151436
- Smallest singular value: 0.155096
- Numerical rank: 400
- Relative Gram reconstruction error: 1.55e-15
- U orthogonality error: 4.71e-14

**Mathematical interpretation:**

The MRI matrix has full row rank, indicating 400
linearly independent observations at the selected
numerical tolerance.

The singular values decrease substantially,
suggesting that some directions contribute much
more strongly to the matrix's energy than others.

**Verification:**
All five numerical checks passed.

**Conclusion:**
The singular values and left singular vectors
were successfully computed using the smaller
400 × 400 Gram matrix.

Full reconstruction of X will be verified in Block 13.


# ***Largest singular value.***
# ***Energy explained***

In [5]:

# Energy explained by the largest singular value
energy = singular_values**2

percentage = 100 * energy[0] / energy.sum()

print(f"Largest singular value: {singular_values[0]:.6f}")
print(f"Energy explained: {percentage:.4f}%")


Largest singular value: 591.151436
Energy explained: 73.1302%


## Block 13 — SVD Reconstruction


In [6]:

import numpy as np

print("BLOCK 13 — SVD RECONSTRUCTION")

# Reuse Block 12 results
X_svd = X64

# Construct right singular vectors
Vt = (U.T @ X_svd) / singular_values[:, None]

# Reconstruct original matrix
X_reconstructed = (U * singular_values) @ Vt

# Relative reconstruction error
relative_error = (
    np.linalg.norm(X_svd - X_reconstructed, ord="fro")
    / np.linalg.norm(X_svd, ord="fro")
)

# Orthogonality of right singular vectors
Vt_orthogonality_error = np.linalg.norm(
    Vt @ Vt.T - np.eye(len(singular_values)),
    ord="fro"
)

# Individual pixel reconstruction error
max_absolute_error = np.max(
    np.abs(X_svd - X_reconstructed)
)

print("\nMATRIX DIMENSIONS")
print("X shape:", X_svd.shape)
print("U shape:", U.shape)
print("Singular values shape:", singular_values.shape)
print("Vt shape:", Vt.shape)
print("Reconstructed X shape:", X_reconstructed.shape)

print("\nRECONSTRUCTION RESULTS")
print("Relative reconstruction error:", relative_error)
print("Maximum absolute pixel error:", max_absolute_error)
print("Vt orthogonality error:", Vt_orthogonality_error)

print("\nFINAL VERIFICATION")

checks = {
    "Reconstruction dimensions match":
        X_reconstructed.shape == X_svd.shape,
    "Relative reconstruction error < 1e-10":
        relative_error < 1e-10,
    "Maximum absolute error < 1e-10":
        max_absolute_error < 1e-10,
    "Right singular vectors orthogonal":
        Vt_orthogonality_error < 1e-8,
}

for name, passed in checks.items():
    print(f"{name}: {passed}")

assert all(checks.values()), (
    "Block 13 verification failed."
)

print("\nALL BLOCK 13 CHECKS PASSED:", all(checks.values()))


BLOCK 13 — SVD RECONSTRUCTION

MATRIX DIMENSIONS
X shape: (400, 16384)
U shape: (400, 400)
Singular values shape: (400,)
Vt shape: (400, 16384)
Reconstructed X shape: (400, 16384)

RECONSTRUCTION RESULTS
Relative reconstruction error: 1.3311357021119366e-15
Maximum absolute pixel error: 3.885780586188048e-15
Vt orthogonality error: 4.7252143813416025e-11

FINAL VERIFICATION
Reconstruction dimensions match: True
Relative reconstruction error < 1e-10: True
Maximum absolute error < 1e-10: True
Right singular vectors orthogonal: True

ALL BLOCK 13 CHECKS PASSED: True



### Block 13 — SVD Reconstruction and Verification

**Objective:** Reconstruct the original MRI matrix using
the complete reduced Singular Value Decomposition.

**Matrix dimensions:**
- Original X: 400 × 16,384
- U: 400 × 400
- Vᵀ: 400 × 16,384
- Reconstructed X: 400 × 16,384

**Numerical results:**
- Relative reconstruction error: 1.3311e-15
- Maximum absolute pixel error: 3.8858e-15
- Right singular vector orthogonality error: 4.7252e-11

**Verification:**
All four numerical checks passed.

**Conclusion:**
The complete reduced SVD accurately reconstructs
the original MRI matrix with negligible numerical error.

This validates the decomposition before applying
truncated SVD for dimensionality reduction.


## Block 14 — SVD Dimensionality Reduction and Low-Rank Approximation


In [7]:

import numpy as np

print("BLOCK 14 — SVD DIMENSIONALITY REDUCTION")

# Original MRI matrix
X_svd = X64
n, p = X_svd.shape

# Retain 50 singular directions
k = 50

# Compressed representation
Z = U[:, :k] * singular_values[:k]

# Low-rank reconstruction
X_k = Z @ Vt[:k, :]

# Retained matrix energy
energy = singular_values**2
retained_energy = 100 * energy[:k].sum() / energy.sum()

# Reconstruction error
reconstruction_error = (
    np.linalg.norm(X_svd - X_k, ord="fro")
    / np.linalg.norm(X_svd, ord="fro")
)

# Theoretical error from singular values
theoretical_error = np.sqrt(
    energy[k:].sum() / energy.sum()
)

# Numerical verification
checks = {
    "Compressed matrix dimensions":
        Z.shape == (n, k),
    "Reconstructed matrix dimensions":
        X_k.shape == X_svd.shape,
    "All compressed values finite":
        np.isfinite(Z).all(),
    "Rank of approximation <= 50":
        np.linalg.matrix_rank(Z) <= k,
    "Reconstruction error matches theory":
        np.isclose(
            reconstruction_error,
            theoretical_error,
            atol=1e-8
        ),
}

print("\nMATRIX DIMENSIONS")
print("Original X:", X_svd.shape)
print("Compressed Z:", Z.shape)
print("Approximation X_k:", X_k.shape)

print("\nFIRST FIVE ROWS AND COLUMNS OF Z")
print(Z[:5, :5])

print("\nENERGY AND ERROR")
print(f"Singular directions retained: {k}")
print(f"Retained energy: {retained_energy:.4f}%")
print(f"Measured reconstruction error: {reconstruction_error:.8f}")
print(f"Theoretical reconstruction error: {theoretical_error:.8f}")

print("\nFINAL VERIFICATION")
for name, passed in checks.items():
    print(f"{name}: {passed}")

assert all(checks.values()), "Block 14 verification failed."

print("\nALL BLOCK 14 CHECKS PASSED:", all(checks.values()))


BLOCK 14 — SVD DIMENSIONALITY REDUCTION

MATRIX DIMENSIONS
Original X: (400, 16384)
Compressed Z: (400, 50)
Approximation X_k: (400, 16384)

FIRST FIVE ROWS AND COLUMNS OF Z
[[19.90529766 -3.02698094 -4.29473287 -6.45736623 -2.18845076]
 [30.13756309  1.22179996  6.86765672 -2.27276412  1.82475767]
 [21.10968886 -3.26857552  2.54405467 -0.62996087 -0.5379116 ]
 [31.39291106  6.21174588  0.06658498 -1.89302634  3.29285507]
 [21.32093928 -3.24013742  2.51299633 -0.60438996 -0.70827367]]

ENERGY AND ERROR
Singular directions retained: 50
Retained energy: 90.7904%
Measured reconstruction error: 0.30347301
Theoretical reconstruction error: 0.30347301

FINAL VERIFICATION
Compressed matrix dimensions: True
Reconstructed matrix dimensions: True
All compressed values finite: True
Rank of approximation <= 50: True
Reconstruction error matches theory: True

ALL BLOCK 14 CHECKS PASSED: True



### Block 14 — SVD Dimensionality Reduction

**Objective:** Reduce MRI feature dimensionality using truncated SVD.

**Results:**
- Original matrix X: 400 × 16,384
- Compressed matrix Z: 400 × 50
- Reconstructed matrix X₅₀: 400 × 16,384
- Feature reduction: 99.69%
- Retained energy: 90.7904%
- Relative reconstruction error: 0.30347301

**Conclusion:**
SVD reduced each MRI image from 16,384 original
features to 50 numerical features while preserving
90.79% of the uncentered matrix energy.

All five numerical verification checks passed.


In [8]:

import numpy as np

print("BLOCK 15 — SVD FINAL EVALUATION")

# Energy from singular values
energy = singular_values**2
total_energy = energy.sum()
cumulative_energy = np.cumsum(energy) / total_energy

# Energy targets
targets = [0.90, 0.95, 0.99]

print("\nENERGY TARGETS")

results = {}

for target in targets:
    k = np.searchsorted(cumulative_energy, target) + 1

    actual_energy = 100 * cumulative_energy[k - 1]

    theoretical_error = np.sqrt(
        energy[k:].sum() / total_energy
    )

    results[target] = (k, actual_energy, theoretical_error)

    print(f"\nTarget energy: {100 * target:.0f}%")
    print(f"Required singular directions: {k}")
    print(f"Compressed Z shape: ({X64.shape[0]}, {k})")
    print(f"Actual retained energy: {actual_energy:.4f}%")
    print(f"Relative reconstruction error: {theoretical_error:.8f}")

# Final numerical verification
checks = {
    "Energy is normalized":
        np.isclose(cumulative_energy[-1], 1.0),
    "Cumulative energy is nondecreasing":
        np.all(np.diff(cumulative_energy) >= -1e-12),
    "Energy thresholds reached":
        all(
            cumulative_energy[k - 1] >= target - 1e-12
            for target, (k, _, _) in results.items()
        ),
    "Required ranks increase with targets":
        all(
            results[targets[i]][0] <= results[targets[i + 1]][0]
            for i in range(len(targets) - 1)
        ),
}

print("\nFINAL VERIFICATION")
for name, passed in checks.items():
    print(f"{name}: {passed}")

assert all(checks.values()), "Block 15 verification failed."

print("\nALL BLOCK 15 CHECKS PASSED:", all(checks.values()))


BLOCK 15 — SVD FINAL EVALUATION

ENERGY TARGETS

Target energy: 90%
Required singular directions: 44
Compressed Z shape: (400, 44)
Actual retained energy: 90.0750%
Relative reconstruction error: 0.31504002

Target energy: 95%
Required singular directions: 106
Compressed Z shape: (400, 106)
Actual retained energy: 95.0181%
Relative reconstruction error: 0.22320209

Target energy: 99%
Required singular directions: 257
Compressed Z shape: (400, 257)
Actual retained energy: 99.0089%
Relative reconstruction error: 0.09955648

FINAL VERIFICATION
Energy is normalized: True
Cumulative energy is nondecreasing: True
Energy thresholds reached: True
Required ranks increase with targets: True

ALL BLOCK 15 CHECKS PASSED: True


***Block 15 — SVD Final Evaluation***


### Block 15 — SVD Energy-Based Component Selection

**Objective:** Determine the minimum number of singular
directions required to preserve 90%, 95%, and 99%
of the original matrix energy.

**Method:**
- Calculate cumulative explained energy.
- Identify the minimum rank k for each energy target.
- Determine compressed matrix dimensions (400 × k).
- Calculate theoretical relative reconstruction errors.

**Expected interpretation:**
Higher energy retention generally requires more
singular directions and produces smaller reconstruction
errors.

**Conclusion:**
Energy-based rank selection provides a quantitative
method for balancing dimensionality reduction and
reconstruction accuracy.

**Status:** Awaiting Block 15 numerical results.



### Block 15 — SVD Final Evaluation

**Objective:** Determine the minimum number of singular
directions required for different energy targets.

**Results:**

| Target | Components | Actual Energy | Relative Error |
|---|---:|---:|---:|
| 90% | 44 | 90.0750% | 0.31504002 |
| 95% | 106 | 95.0181% | 0.22320209 |
| 99% | 257 | 99.0089% | 0.09955648 |

**Mathematical interpretation:**

Higher retained energy requires more singular
directions and produces smaller reconstruction errors.

At the 90% target, each MRI image is represented
by 44 numerical features instead of 16,384.

**Verification:**
All four numerical checks passed.

**Conclusion:**
Truncated SVD provides a mathematically verified
approach to dimensionality reduction and low-rank
approximation of MRI data.

Energy preservation does not necessarily guarantee
preservation of clinically relevant information.
